---
format:
  pdf:
    include-in-header:
      text: |
        \usepackage{pdflscape}
    toc: false
    number-sections: true
    tbl-cap-location: bottom
execute:
  echo: false
  enabled: true
---

\begin{titlepage}
\centering

\vspace*{2cm}

\includegraphics[width=0.3\textwidth]{ncl_logo.png}

\vspace{1.5cm}

{\Huge\bfseries 2026 DEFRA, Local Council and Urban Observatory Sensor Data Analysis\par}

\vspace{2cm}

{\Large Axa-Maria Laaperi, Dr Alexandra Svalova, Dr Graeme Sarson, Prof Anvar Shukurov\par}

\vspace{0.5cm}

School of Mathematics, Statistics and Physics\\
Newcastle University

\vfill

July 2026

\end{titlepage}

\newpage

## Executive Summary {#sec-summary}

After becoming aware of NCC's restrictions in only using calibrated DEFRA-approved and -owned AURN instrumentation and locally-managed automatic monitoring sensors, we have produced a short data analysis summary in order to determine the validity of the Urban Observatory equipment. The Urban Observatory (UO) has a collection of precison MONITOR sensors. The UO MESH sensors are not calibrated and not approved by DEFRA and therefore NCC cannot use them. For use in a multi-sensor spatial model as planned for the proposed app, we require as many sensors as possible and therefore verification that the UO MONITOR sensor data agrees with that of DEFRA and AURN sensors is needed. 

For the purposes of this report we will use the following acronyms:

- 2 DEFRA-owned and approved AURN sensors (DEFRA)
- 5 Locally managed automatic monitoring sensors (Local)
- 5 UO precision MONITOR sensors (UO-Mon)
- UO Mesh sensors (UO-Mesh)

We have highlighted the following as areas for priory action/attention:

- To validate the UO-Mon sensors we compared them to their closest DEFRA/Local sensors, all UO-Mon sensors compared had very good agreement with their closest DEFRA/Local sensor. Some UO-Mon sensors were compared with further (in distance) DEFRA/Local sensors in order to compare all DEFRA/Local sensors. We found increase in distance to change the relationship between sensors to resemble an exponenital. This was not fully consistent and we would appreciate some expert UO opinion on this matter.
- For PM2.5 data, the UO-Mon and UO-Mesh sensors have low agreement. We will proceed with using the UO-Mon sensors only.
- Some UO-Mon sensors have faulty wind direction and speed data, shown as constant over the course of a day, week and month, we have taken some decommisoned sensor data from 2025 to produce analysis.
- DEFRA Newcastle Centre sensor is systematically rounding PM2.5 values to nearest integer, this will be brought to NCC's attention.

The report is structured as follows:

- Long time interval analysis, March 2026, comparing PM2.5 concentrations from closest precision sensors (DEFRA/Local/UO-Mon) to UO-Mesh sensors and closest DEFRA/Local sensors to UO-Mon sensors. March 2026 reasonably demonstrates a longer time period where these sensors could differ in 15min - 1hr intervalled collection. We chose month that was relatively recent with minimal missing sensors and values. Caveat: there is a short time period between 19th-21st where data is missing from a DEFRA sensors.
- Short time interval analysis, 30th March - 5th April 2026, comparing PM2.5 concentrations to NOx and NO2 concentrations from same sensors for DEFRA, Local and UO-Mon sensors. We chose a shorter time scale for this analysis as NOx and NO2 concentrations degrade very fast, so only a short time scale is appropriate for their analysis case. This has not been outlined in this shorter report however the analysis is ready for distribution if it is of interest.

Note:

- We have further analysis on wind speed/direction, temperature and altitude relationships with PM2.5 that is not outlined in this report as well as UO-Mesh comparisons.
- When exaining wind effects, comparing speed and direction to concentration in 2026 was not possible as stated in the above reasoning. With 2025 data, we take old, decommisioned sensors and perform similar analysis with closest UO-Mon sensors and Local sensors. DEFRA sensors were not considered as they are situated further away.

\newpage


\tableofcontents
\newpage

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
import seaborn as sns
import uo_pyfetch
import datetime
from IPython.display import display, Markdown
import plotly.graph_objects as go
from geopy.distance import geodesic
from statsmodels.tsa.stattools import acf
import math
from shapely.geometry import Point, LineString
import rasterio
import warnings
import matplotlib.cm as cm
import matplotlib.colors as colors
warnings.filterwarnings("ignore", category=UserWarning)

def to_float64(X):
    """
    Convert input data to float64.

    GPflow requires input data to be in float64 format.
    This function ensures compatibility and prevents dtype errors.
    """
    return np.asarray(X, dtype=np.float64)

uo_name_changes = pd.read_csv("naming.csv")
uo_mapping = dict(
        zip(uo_name_changes["UO_MESH_MON"],
            uo_name_changes["new_name"])
    )

In [ ]:
def closest_sensor(locations_1, locations_2):
    
    results = []
        
    for _, row_1 in locations_1.iterrows():

        coords_1 = (
            row_1['Lat'],
            row_1['Long']
        )

        for _, row_2 in locations_2.iterrows():

            coords_2 = (
                row_2['Lat'],
                row_2['Long']
            )

            distance_km = geodesic(coords_1, coords_2).km

            results.append({
                'Sensor 1': row_1['Sensor_Name'],
                'Sensor 2': row_2['Sensor_Name'],
                'Distance km': distance_km
            })

    distance_df = pd.DataFrame(results)

    closest_sensors_df = (
        distance_df.loc[
            distance_df.groupby('Sensor 1')['Distance km'].idxmin()
        ]
    )
        
    return(closest_sensors_df)

In [ ]:
def timeframe_PM25_2026(date_start, date_end):

    PM25_2026 = pd.read_csv("2026uptoMay27-PM25-UO.csv") 
    DEFRA_2026 = pd.read_csv("2026uptoMay13-PM25-DEFRA.csv") 
    local_2026 = pd.read_csv("2026uptoMay13-PM25-local.csv")
    precision_list = [DEFRA_2026, local_2026] 

    PM25_2026["Timestamp"] = pd.to_datetime(PM25_2026["Timestamp"], errors="coerce")

    for df in precision_list: 
        date = df['Date'].astype(str) 
        time = df['Time'].astype(str) 
        mask_24 = time.str.startswith('24:') 
        # fix time first 
        time_fixed = time.str.replace(r'^24:', '00:', regex=True) 
        # combine as strings 
        combined = date + ' ' + time_fixed 
        # parse AFTER fixing 
        ts = pd.to_datetime(combined, dayfirst=True) 
        # now shift ONLY those originally with 24:00 
        ts = ts + pd.to_timedelta(mask_24.astype(int), unit='D') 
        df['Timestamp'] = ts 
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce") 

    uo = PM25_2026[
        (PM25_2026["Timestamp"] >= date_start) &
        (PM25_2026["Timestamp"] < date_end)
    ]

    defra = DEFRA_2026[
        (DEFRA_2026["Timestamp"] >= date_start) &
        (DEFRA_2026["Timestamp"] < date_end)
    ]

    local = local_2026[
        (local_2026["Timestamp"] >= date_start) &
        (local_2026["Timestamp"] < date_end)
    ]

    defra["Sensor_Name"] = "DEFRA-" + defra["Sensor_Name"].astype(str)
    local["Sensor_Name"] = "Local-" + local["Sensor_Name"].astype(str)
    uo["Sensor_Name"] = (
        uo["Sensor_Name"]
        .map(uo_mapping)
        .fillna(uo["Sensor_Name"])
    )

    sensor_locations = (uo[['Sensor_Name', 'Sensor_Centroid_Longitude', 'Sensor_Centroid_Latitude']] 
                    .dropna() 
                    .drop_duplicates(subset='Sensor_Name') 
                    .reset_index(drop=True) 
                    .rename(columns={'Sensor_Centroid_Longitude': 'Long'}) 
                    .rename(columns={'Sensor_Centroid_Latitude': 'Lat'}) 
                    ) 

    mesh_locations = (sensor_locations[sensor_locations['Sensor_Name']
                    .str.contains('Mesh', na=False)]
                    .reset_index(drop=True) 
                    .drop_duplicates(subset='Sensor_Name') 
                    ) 

    monitor_locations = (sensor_locations[sensor_locations['Sensor_Name']
                                        .str.contains('Mon', na=False)] 
                                        .reset_index(drop=True) 
                                        .drop_duplicates(subset='Sensor_Name') 
                                        ) 

    defra_locations = (defra[['Sensor_Name', 'Long', 'Lat']] 
                    .dropna() 
                    .drop_duplicates(subset='Sensor_Name') 
                    .reset_index(drop=True) 
                    ) 

    local_locations = (local[['Sensor_Name', 'Long', "Lat"]] 
                    .dropna() 
                    .drop_duplicates(subset='Sensor_Name') 
                    .reset_index(drop=True)
                    ) 

    precision_locations = pd.concat(
        [
            defra_locations,
            local_locations,
            monitor_locations
        ],
        ignore_index=True
    )

    all_defra_locations = pd.concat(
        [
            defra_locations,
            local_locations
        ],
        ignore_index=True
    )

    closest_precision_mesh = closest_sensor(
        precision_locations,
        mesh_locations
    )

    closest_defra_monitor = closest_sensor(
        all_defra_locations,
        monitor_locations
    )

    closest_mesh_monitor = closest_sensor(
        mesh_locations,
        monitor_locations
    )

    return {
        'uo': uo,
        'defra': defra,
        'local': local,
        'mesh_locations': mesh_locations,
        'monitor_locations': monitor_locations,
        'defra_locations': defra_locations,
        'local_locations': local_locations,
        'precision_locations': precision_locations,
        'all_defra_locations': all_defra_locations,
        'closest_precision_mesh': closest_precision_mesh,
        'closest_defra_monitor': closest_defra_monitor,
        'closest_mesh_monitor': closest_mesh_monitor
    }

In [ ]:
def timeframe_NO_2026(date_start, date_end):

    NOx_2026 = pd.read_csv("2026uptoMay27-NOx-UO.csv") 
    DEFRA_2026_NO = pd.read_csv("2026uptoMay13-NO-DEFRA.csv") 
    local_2026_NO = pd.read_csv("2026uptoMay13-NO-local.csv")
    precision_list = [DEFRA_2026_NO, local_2026_NO] 

    NOx_2026["Timestamp"] = pd.to_datetime(NOx_2026["Timestamp"], errors="coerce")

    for df in precision_list: 
        date = df['Date'].astype(str) 
        time = df['Time'].astype(str) 
        mask_24 = time.str.startswith('24:') 
        # fix time first 
        time_fixed = time.str.replace(r'^24:', '00:', regex=True) 
        # combine as strings 
        combined = date + ' ' + time_fixed 
        # parse AFTER fixing 
        ts = pd.to_datetime(combined, dayfirst=True) 
        # now shift ONLY those originally with 24:00 
        ts = ts + pd.to_timedelta(mask_24.astype(int), unit='D') 
        df['Timestamp'] = ts 
        df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

    uo = NOx_2026[
        (NOx_2026["Timestamp"] >= date_start) &
        (NOx_2026["Timestamp"] < date_end)
    ]

    defra = DEFRA_2026_NO[
        (DEFRA_2026_NO["Timestamp"] >= date_start) &
        (DEFRA_2026_NO["Timestamp"] < date_end)
    ]

    local = local_2026_NO[
        (local_2026_NO["Timestamp"] >= date_start) &
        (local_2026_NO["Timestamp"] < date_end)
    ]

    defra["Sensor_Name"] = "DEFRA-" + defra["Sensor_Name"].astype(str)
    local["Sensor_Name"] = "Local-" + local["Sensor_Name"].astype(str)
    uo["Sensor_Name"] = (
        uo["Sensor_Name"]
        .map(uo_mapping)
        .fillna(uo["Sensor_Name"])
    )
    
    return {
        'uo': uo,
        'defra': defra,
        'local': local
    }

In [ ]:
def timeframe_PM25_wind_2025(date_start, date_end):

    PM25_2025 = pd.read_csv('2025-PM25-UO.csv')
    PM25_2025_local = pd.read_csv('2025March-PM25-local.csv')
    westdenton_2025_wind = pd.read_csv('2025-wind-UO_westdenton.csv')
    birtley_2025_wind = pd.read_csv('2025-wind-UO_birtley.csv')

    PM25_2025["Timestamp"] = pd.to_datetime(PM25_2025["Timestamp"], errors="coerce")
    westdenton_2025_wind["Timestamp"] = pd.to_datetime(westdenton_2025_wind["Timestamp"], errors="coerce")
    birtley_2025_wind["Timestamp"] = pd.to_datetime(birtley_2025_wind["Timestamp"], errors="coerce")
    
    date = PM25_2025_local['Date'].astype(str) 
    time = PM25_2025_local['Time'].astype(str) 
    mask_24 = time.str.startswith('24:') 
    # fix time first 
    time_fixed = time.str.replace(r'^24:', '00:', regex=True) 
    # combine as strings 
    combined = date + ' ' + time_fixed 
    # parse AFTER fixing 
    ts = pd.to_datetime(combined, dayfirst=True) 
    # now shift ONLY those originally with 24:00 
    ts = ts + pd.to_timedelta(mask_24.astype(int), unit='D') 
    PM25_2025_local['Timestamp'] = ts 
    PM25_2025_local["Timestamp"] = pd.to_datetime(PM25_2025_local["Timestamp"], errors="coerce")

    uo = PM25_2025[
        (PM25_2025["Timestamp"] >= date_start) &
        (PM25_2025["Timestamp"] < date_end)
    ]

    local = PM25_2025_local[
        (PM25_2025_local["Timestamp"] >= date_start) &
        (PM25_2025_local["Timestamp"] < date_end)
    ]

    wd = westdenton_2025_wind[
        (westdenton_2025_wind["Timestamp"] >= date_start) &
        (westdenton_2025_wind["Timestamp"] < date_end)
    ]

    birt = birtley_2025_wind[
        (birtley_2025_wind["Timestamp"] >= date_start) &
        (birtley_2025_wind["Timestamp"] < date_end)
    ]

    wind = pd.concat(
        [wd, birt]
    )
    
    local["Sensor_Name"] = "Local-" + local["Sensor_Name"].astype(str)
    uo["Sensor_Name"] = (
        uo["Sensor_Name"]
        .map(uo_mapping)
        .fillna(uo["Sensor_Name"])
    )
    rename_wind_sensors = {
    "PER_EMLFLOOD_UO-BIRTLEYFS": "UO-WIND-birtley",
    "PER_EMLFLOOD_UO-WDENTONFS": "UO-WIND-wdenton"
    }
    wind["Sensor_Name"] = wind["Sensor_Name"].replace(rename_wind_sensors)

    wind_locations = (wind[['Sensor_Name', 'Sensor_Centroid_Longitude', 'Sensor_Centroid_Latitude']]
                .dropna()
                .drop_duplicates(subset='Sensor_Name')
                .rename(columns={'Sensor_Centroid_Longitude': 'Long',
                                'Sensor_Centroid_Latitude': 'Lat'
                                })
                )
    
    monitor_locations = (uo[uo['Sensor_Name']
                                        .str.contains('Mon', na=False)] 
                                        .reset_index(drop=True) 
                                        .drop_duplicates(subset='Sensor_Name')
                                        .rename(columns={'Sensor_Centroid_Longitude': 'Long',
                                                         'Sensor_Centroid_Latitude': 'Lat'})
                                        ) 
    
    local_locations = (local[['Sensor_Name', 'Long', "Lat"]] 
                    .dropna() 
                    .drop_duplicates(subset='Sensor_Name') 
                    .reset_index(drop=True)
                    )

    closest_wind_monitor = closest_sensor(
        wind_locations,
        monitor_locations
    )

    closest_wind_local = closest_sensor(
        wind_locations,
        local_locations
    )

    return {
        'uo': uo,
        'wind': wind,
        'local': local,
        'wind_locations': wind_locations,
        'monitor_locations': monitor_locations,
        'local_locations': local_locations,
        'closest_wind_monitor': closest_wind_monitor,
        'closest_wind_local': closest_wind_local
    }

In [ ]:
march_2026 = timeframe_PM25_2026("2026-03-01", "2026-03-31")

defra_names = set(march_2026['defra_locations']['Sensor_Name'].unique())
local_names = set(march_2026['local_locations']['Sensor_Name'].unique())
monitor_names = set(march_2026['monitor_locations']['Sensor_Name'].unique()) 
mesh_names = set(march_2026['mesh_locations']['Sensor_Name'].unique()) 

In [ ]:
march_week_2026 = timeframe_PM25_2026("2026-03-30","2026-04-05")

## Sensor locations {#sec-data}

The three maps in @fig-sensor-maps-1 show closest precision sensors (DEFRA/Local/UO-Mon) to UO-Mesh sensors, closest DEFRA/Local sensors to UO-Mon sensors. These pairs will be used to analyse correlations between closest sensors and validate UO-Mon sensors. The distances (km) between these pairs are stated in @sec-pm25-close correlational analysis.

In [ ]:
def plot_sensor_pairs_map(
    pair_df,
    locations_1,
    locations_2,
    all_locations=None,
    label_1='Sensor 1',
    label_2='Sensor 2',
    title='Sensor Pair Map',
    figsize=(15, 15),
    ax=None
):

    merged = (
        pair_df
        .merge(
            locations_1[
                ['Sensor_Name', 'Long', 'Lat']
            ],
            left_on='Sensor 1',
            right_on='Sensor_Name',
            how='left'
        )
        .rename(columns={
            'Long': 'Long_1',
            'Lat': 'Lat_1'
        })
        .drop(columns='Sensor_Name')
    )

    merged = (
        merged
        .merge(
            locations_2[
                ['Sensor_Name', 'Long', 'Lat']
            ],
            left_on='Sensor 2',
            right_on='Sensor_Name',
            how='left'
        )
        .rename(columns={
            'Long': 'Long_2',
            'Lat': 'Lat_2'
        })
        .drop(columns='Sensor_Name')
    )

    line_geoms = []

    for _, row in merged.iterrows():

        if (
            pd.isna(row['Long_1'])
            or
            pd.isna(row['Long_2'])
        ):
            continue

        line_geoms.append(
            LineString([
                (
                    row['Long_1'],
                    row['Lat_1']
                ),
                (
                    row['Long_2'],
                    row['Lat_2']
                )
            ])
        )

    lines_gdf = gpd.GeoDataFrame(
        geometry=line_geoms,
        crs='EPSG:4326'
    )

    paired_1 = gpd.GeoDataFrame(
        merged,
        geometry=gpd.points_from_xy(
            merged['Long_1'],
            merged['Lat_1']
        ),
        crs='EPSG:4326'
    )

    paired_2 = gpd.GeoDataFrame(
        merged,
        geometry=gpd.points_from_xy(
            merged['Long_2'],
            merged['Lat_2']
        ),
        crs='EPSG:4326'
    )

    background_gdf = None

    if all_locations is not None:

        paired_names = set(
            pair_df['Sensor 1']
        ).union(
            set(pair_df['Sensor 2'])
        )

        background = (
            all_locations[
                ~all_locations[
                    'Sensor_Name'
                ].isin(paired_names)
            ]
        )

        background_gdf = gpd.GeoDataFrame(
            background,
            geometry=gpd.points_from_xy(
                background['Long'],
                background['Lat']
            ),
            crs='EPSG:4326'
        )

    paired_1 = paired_1.to_crs(3857)
    paired_2 = paired_2.to_crs(3857)
    lines_gdf = lines_gdf.to_crs(3857)

    if background_gdf is not None:
        background_gdf = (
            background_gdf
            .to_crs(3857)
        )
        
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)

    # other sensors
    if background_gdf is not None:

        background_gdf.plot(
            ax=ax,
            markersize=20,
            alpha=0.35,
            label='Other sensors'
        )

    # connecting lines
    lines_gdf.plot(
        ax=ax,
        linewidth=1.5,
        alpha=0.7,
        label='Pair links'
    )

    # paired sensors
    paired_1.plot(
        ax=ax,
        markersize=80,
        label=label_1
    )

    paired_2.plot(
        ax=ax,
        markersize=80,
        marker='^',
        label=label_2
    )

    # basemap
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron
    )

    ax.set_title(title, fontsize=16)
    ax.legend(prop={'size': 14})
    ax.set_axis_off()

In [ ]:
#| label: fig-sensor-maps-1
#| fig-cap: "Maps of (left to right) closest precision sensors (DEFRA/Local/UO-Mon) to UO-Mesh sensors, closest DEFRA/Local sensors to UO-Mon sensors, and closest UO-Mon sensors to UO-Mesh sensors."

fig = plt.figure(figsize=(20, 14))

gs = fig.add_gridspec(2, 2)

axes = [
    fig.add_subplot(gs[0, 0]),
    fig.add_subplot(gs[0, 1]),
    fig.add_subplot(gs[1, :])
]

all_sensors = pd.concat(
    [
        march_2026['precision_locations'],
        march_2026['mesh_locations']
    ],
    ignore_index=True
)

plot_sensor_pairs_map(
    march_2026['closest_precision_mesh'],
    march_2026['precision_locations'],
    march_2026['mesh_locations'],
    all_locations=all_sensors,
    label_1='Precision (DEFRA/Local/UO-Mon)',
    label_2='UO-Mesh',
    title='Closest Precision to UO-Mesh Sensors',
    ax=axes[0]
)

all_sensors = pd.concat(
    [
        march_2026['all_defra_locations'],
        march_2026['monitor_locations']
    ],
    ignore_index=True
)

plot_sensor_pairs_map(
    march_2026['closest_defra_monitor'],
    march_2026['all_defra_locations'],
    march_2026['monitor_locations'],
    all_locations=all_sensors,
    label_1='DEFRA/Local',
    label_2='UO-Mon',
    title='Closest DEFRA/Local to UO-Mon Sensors',
    ax=axes[1]
)

all_sensors = pd.concat(
    [
        march_2026['monitor_locations'],
        march_2026['mesh_locations']
    ],
    ignore_index=True
)

plot_sensor_pairs_map(
    march_week_2026['closest_mesh_monitor'],
    march_week_2026['mesh_locations'],
    march_week_2026['monitor_locations'],
    all_locations=all_sensors,
    label_1='UO-Mesh',
    label_2='UO-Mon',
    title='Closest UO-Mon to UO-Mesh Sensors',
    ax=axes[2]
)

plt.tight_layout()
plt.show()

In [ ]:
closest_pairs = (
    march_2026['closest_defra_monitor']
    .sort_values('Distance km')
    .drop_duplicates(subset='Sensor 2', keep='first')
)

repeat_pairs = (
    march_2026['closest_defra_monitor']
    .loc[
        ~march_2026['closest_defra_monitor'].index.isin(closest_pairs.index)
    ]
)

\newpage

## Close sensor comparisons {#sec-pm25-close}

We used March 2026 to look at correlation comparisons between the closest DEFRA/Local sensors to UO-Mon sensors. March was chosen as the dataset had reliable consistent data, i.e. minimal missing values and missing sensor data however there was still a portion of the data missing around the 19th-21st from the DEFRA sensors.

@fig-pm25-closest-defra-local-mon shows closest UO-Mon sensors to DEFRA/Local sensors, scatter plots and time series, where only the closest UO-Mon sensor is included for each DEFRA/Local sensor. This hopes to validate the UO-Mon sensor data and from both sets of plots we see very good agreement between the sensors. @fig-pm25-closest-defra-local-mon-ex shows the same comparison however the UO-Mon sensors under consideration (UO-Mon Gateshead Tyne Bridge and UO-Mon RVI) have already been compared to other DEFRA/Local sensors in @fig-pm25-closest-defra-local-mon that are much closer, we see worse agreement for two of the sensor comparisons here which is expected due to the distance. A summary table of correlational information and distances between sensors is given in @tbl-corr-defra-local-mon from which we see high correlation between all close sensors. We use both Pearson's and Spearman's correlation coefficents. Pearson's evaluates linear relationships and proportional change between variables while Spearman's monotonic relationships which dicerns if variables are changing together, not necessarily at a constant rate as Pearson's would let us assume. It is important to remember that other relationship (non-linear) might also exist, we have examined other external factors that are not included in this report but would be available if they are of interest.

In [ ]:
summary_results = []

def make_valid_rows(pair_df):

    valid_rows = []

    for _, row in pair_df.iterrows():

        precision_name = row['Sensor 1']
        monitor_name = row['Sensor 2']

        if precision_name in defra_names:
            precision_df = march_2026['defra']
        elif precision_name in local_names:
            precision_df = march_2026['local']

        precision = (
            precision_df[
                precision_df['Sensor_Name'] == precision_name
            ][['Timestamp', 'PM2.5']]
            .sort_values('Timestamp')
            .rename(columns={'PM2.5': 'precision_pm25'})
        )

        precision['precision_pm25'] = pd.to_numeric(
            precision['precision_pm25'],
            errors='coerce'
        )

        monitor = (
            march_2026['uo'][
                march_2026['uo']['Sensor_Name'] == monitor_name
            ][['Timestamp', 'Value']]
            .sort_values('Timestamp')
            .rename(columns={'Value': 'monitor_pm25'})
        )

        monitor['monitor_pm25'] = pd.to_numeric(
            monitor['monitor_pm25'],
            errors='coerce'
        )

        merged = pd.merge_asof(
            precision,
            monitor,
            on='Timestamp',
            direction='nearest',
            tolerance=pd.Timedelta('1min')
        ).dropna(subset=['precision_pm25', 'monitor_pm25'])

        if not merged.empty:
            valid_rows.append((row, precision_name, monitor_name, merged))

    return valid_rows

closest_valid_rows = make_valid_rows(closest_pairs)
repeat_valid_rows = make_valid_rows(repeat_pairs)

In [ ]:
#| label: fig-pm25-closest-defra-local-mon
#| fig-cap: "Comparison of PM2.5 concentrations measured by each DEFRA/Local sensor and its closest UO-Mon sensor over the study period. Only the closest UO-Mon sensor is included for each DEFRA/Local sensor. Very good agreement between all sensor comparisons."

n_pairs = len(closest_valid_rows)

ncols_pairs = 2
nrows_pairs = math.ceil(n_pairs / ncols_pairs)

fig, axes = plt.subplots(
    nrows=nrows_pairs,
    ncols=ncols_pairs * 2,   # 2 plots per pair
    figsize=(20, 4.5 * nrows_pairs),
    constrained_layout=True
)

for i in range(n_pairs, nrows_pairs * ncols_pairs):
        pair_row = i // ncols_pairs
        pair_col = i % ncols_pairs

        ax1 = axes[pair_row, pair_col * 2]
        ax2 = axes[pair_row, pair_col * 2 + 1]

        ax1.set_visible(False)
        ax2.set_visible(False)

axes = np.atleast_2d(axes)

scatter_ref = None

for i, (row, precision_name, monitor_name, merged) in enumerate(closest_valid_rows):

    pair_row = i // ncols_pairs
    pair_col = i % ncols_pairs

    ax1 = axes[pair_row, pair_col * 2]
    ax2 = axes[pair_row, pair_col * 2 + 1]

    # Pearson correlation
    pearson_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'DEFRA/Local': precision_name,
    'UO-Mon': monitor_name,
    'Distance km': row['Distance km'],
    'Pearsons coeff': pearson_corr,
    'Spearmans coeff': spearman_corr,
    })  

    scatter_ref = ax1.scatter(
        merged['precision_pm25'],
        merged['monitor_pm25'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged[['precision_pm25', 'monitor_pm25']].min().min(),
    merged[['precision_pm25', 'monitor_pm25']].max().max()
    ]

    ax1.plot(lims, lims)

    ax1.set_xlabel('DEFRA/Local PM2.5')
    ax1.set_ylabel('UO-Mon PM2.5')
    ax1.grid(True)

    ax2.plot(
        merged['Timestamp'],
        merged['precision_pm25'],
        label='DEFRA'
    )

    ax2.plot(
        merged['Timestamp'],
        merged['monitor_pm25'],
        label='UO-Mon'
    )

    ax2.set_xlabel('Time')
    ax2.set_ylabel('PM2.5')
    ax2.legend()
    ax2.grid(True)
    ax2.tick_params(axis='x', labelrotation=45)

    ax1.set_title(
    f'{precision_name} vs {monitor_name}', pad=10, x=0.99
    )

#plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.show()

In [ ]:
#| label: fig-pm25-closest-defra-local-mon-ex
#| fig-cap: "Comparison of PM2.5 concentrations measured by DEFRA/Local sensors and additional nearby UO-Mon sensors over the study period. These comparisons exclude the nearest UO-Mon sensor shown in @fig-pm25-closest-defra-local-mon and illustrate relationships with other matched UO-Mon sensors. See worse agreement with some non linear relationships which is expected due to the distance."

n_pairs = len(repeat_valid_rows)

ncols_pairs = 2
nrows_pairs = math.ceil(n_pairs / ncols_pairs)

fig, axes = plt.subplots(
    nrows=nrows_pairs,
    ncols=ncols_pairs * 2,   # 2 plots per pair
    figsize=(20, 4.5 * nrows_pairs),
    constrained_layout=True
)

for i in range(n_pairs, nrows_pairs * ncols_pairs):
        pair_row = i // ncols_pairs
        pair_col = i % ncols_pairs

        ax1 = axes[pair_row, pair_col * 2]
        ax2 = axes[pair_row, pair_col * 2 + 1]

        ax1.set_visible(False)
        ax2.set_visible(False)

axes = np.atleast_2d(axes)

scatter_ref = None

for i, (row, precision_name, monitor_name, merged) in enumerate(repeat_valid_rows):

    pair_row = i // ncols_pairs
    pair_col = i % ncols_pairs

    ax1 = axes[pair_row, pair_col * 2]
    ax2 = axes[pair_row, pair_col * 2 + 1]

    # Pearson correlation
    pearson_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr = merged[['precision_pm25', 'monitor_pm25']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'DEFRA/Local': precision_name,
    'UO-Mon': monitor_name,
    'Distance km': row['Distance km'],
    'Pearsons coeff': pearson_corr,
    'Spearmans coeff': spearman_corr,
    })  

    scatter_ref = ax1.scatter(
        merged['precision_pm25'],
        merged['monitor_pm25'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged[['precision_pm25', 'monitor_pm25']].min().min(),
    merged[['precision_pm25', 'monitor_pm25']].max().max()
    ]

    ax1.plot(lims, lims)

    ax1.set_xlabel('DEFRA/Local PM2.5')
    ax1.set_ylabel('UO-Mon PM2.5')
    ax1.grid(True)

    ax2.plot(
        merged['Timestamp'],
        merged['precision_pm25'],
        label='DEFRA'
    )

    ax2.plot(
        merged['Timestamp'],
        merged['monitor_pm25'],
        label='UO-Mon'
    )

    ax2.set_xlabel('Time')
    ax2.set_ylabel('PM2.5')
    ax2.legend()
    ax2.grid(True)
    ax2.tick_params(axis='x', labelrotation=45)

    ax1.set_title(
    f'{precision_name} vs {monitor_name}', pad=10, x=0.99
    )

#plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.show()

Checking the nonlinear relationships seen in @fig-pm25-closest-defra-local-mon-ex, we examined in the agreement between Local-Gateshead A1 Dunston and Local-Gateshead Tyne Bridge since @fig-pm25-closest-defra-local-mon-ex compares with UO-Mon Gateshead Tyne Bridge (in the same location) so we can understand if the disagreement is indeed related to distance and excluding sensor related issues. We should see a similar nonlinear relationship occuring as in @fig-pm25-local-a1-tynebridge which is clearly present.

In [ ]:
#| label: fig-pm25-local-a1-tynebridge
#| fig-cap: "Comparison of PM2.5 concentrations measured by Local-Gateshead A1 Dunston and Local-Gateshead Tyne Bridge. We expect to see a similar non linear relationship to that shown in @fig-pm25-closest-defra-local-mon-ex with Local-Gateshead A1 Dunston and UO-Mon Gateshead Tyne Bridge."
A1 = (
    march_2026['local'][
        march_2026['local']['Sensor_Name'] == 'Local-Gateshead A1 Dunston'
    ][['Timestamp', 'PM2.5']]
    .sort_values('Timestamp')
    .rename(columns={'PM2.5': 'A1_pm25'})
)

A1['A1_pm25'] = pd.to_numeric(
    A1['A1_pm25'],
    errors='coerce'
)

TyneB = (
    march_2026['local'][
        march_2026['local']['Sensor_Name'] == 'Local-Gateshead Tyne Bridge'
    ][['Timestamp', 'PM2.5']]
    .sort_values('Timestamp')
    .rename(columns={'PM2.5': 'TyneB_pm25'})
)

TyneB['TyneB_pm25'] = pd.to_numeric(
    TyneB['TyneB_pm25'],
    errors='coerce'
)

merged = pd.merge_asof(
    A1,
    TyneB,
    on='Timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('1min')
).dropna(subset=['A1_pm25', 'TyneB_pm25'])

fig, ax = plt.subplots(1,2,figsize=(9,4))
ax1 = ax[0]
ax2 = ax[1]

ax1.scatter(merged['A1_pm25'],
           merged['TyneB_pm25'],
           alpha=0.5,
            c='red',
            s = 20)

ax1.set_xlabel('Local-A1 Dunston PM2.5')
ax1.set_ylabel('Local-Tyne Bridge PM2.5')
ax1.set_title('PM2.5')
ax1.grid(True)
ax1.legend()

ax2.plot(
    merged['Timestamp'],
    merged['A1_pm25'],
    label='Gateshead A1 Dunston'
)

ax2.plot(
    merged['Timestamp'],
    merged['TyneB_pm25'],
    label='Gateshead Tyne Bridge'
)

ax2.set_xlabel('Time')
ax2.set_ylabel('PM2.5')
ax2.legend()
ax2.grid(True)
ax2.tick_params(axis='x', labelrotation=45)

ax1.set_title(
f'Local Gateshead A1 Dunston vs Tyne Bridge', pad=10, x=0.99
)

plt.show()

In [ ]:
#| label: tbl-corr-defra-local-mon
#| tbl-cap: "Distances between sensors in km, Pearson's and Spearman's correlations for UO-Mon and closest DEFRA/Local sensors PM2.5 concentrations over long time interval. High correlational coefficents for all close sensors."

summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'DEFRA/Local',
    'UO-Mon',
    'Distance km',
    'Pearsons coeff',
    'Spearmans coeff'
    
]]

summary_df = summary_df.sort_values(
    by='Pearsons coeff',
    ascending=False
)

summary_df[['Distance km', 'Pearsons coeff', 'Spearmans coeff']] = (
    summary_df[['Distance km', 'Pearsons coeff', 'Spearmans coeff']].round(3)
)

Markdown(summary_df.to_markdown(index=False))

In [ ]:
#| label: fig-corr-coeffs
#| fig-cap: "Correlation coefficents of each sensor pair from @fig-pm25-closest-defra-local-mon and @tbl-corr-defra-local-mon with their distance (km)."

fig, ax = plt.subplots(figsize=(4.5,3.5))

ax.scatter(summary_df['Distance km'],
           summary_df['Pearsons coeff'],
           label='Pearson',
           alpha=0.7)

ax.scatter(summary_df['Distance km'],
           summary_df['Spearmans coeff'],
           label='Spearman',
           alpha=0.7)

ax.set_xlabel('Distance km')
ax.set_ylabel('Correlation')
ax.set_ylim(0.7,1)
ax.set_title('Closest UO-Mon to DEFRA/Local sensors')
ax.grid(True)
ax.legend()

plt.show()

\newpage

## NOx and NO2 comparisons {#sec-pm25_NO}

We looked at the last week (30th March-5th April) of March 2026 to examine correlational comparisons of PM2.5 and NOx/NO2 concentrations in the same sensors for all DEFRA, Local and UO-Mon sensors. A shorter time frame was chosen due to NO particle decay occuring much faster than that of PM2.5.

In [ ]:
defra_names = set(march_week_2026['defra_locations']['Sensor_Name'].unique())
local_names = set(march_week_2026['local_locations']['Sensor_Name'].unique())
monitor_names = set(march_week_2026['monitor_locations']['Sensor_Name'].unique()) 
mesh_names = set(march_week_2026['mesh_locations']['Sensor_Name'].unique()) 

### DEFRA sensors {#sec-defra-no}

First NOx and NO2 concentration comparison in the DEFRA sensors to dicern if PM2.5 has any traffic dependency in the same way that NO2 and NOx have. In @fig-pm25-no-defra scatter plots, we see very clearly here the systematic categorisation of the DEFRA Newcastle Centre sensor. Low correlation values can be seen for all in @tbl-pm25-no-defra.

In [ ]:
march_week_2026_NO = timeframe_NO_2026("2026-03-30","2026-04-05")

defra_names = set(march_week_2026_NO['defra']['Sensor_Name'].unique())
local_names = set(march_week_2026_NO['local']['Sensor_Name'].unique())
monitor_names = (march_week_2026_NO['uo'].loc[march_week_2026_NO['uo']['Sensor_Name']
        .str.contains('Mon', na=False),
        'Sensor_Name']
    .unique()
    .tolist()
)

mesh_names = (march_week_2026_NO['uo'].loc[march_week_2026_NO['uo']['Sensor_Name']
        .str.contains('Mesh', na=False),
        'Sensor_Name']
    .unique()
    .tolist()
)

In [ ]:
summary_results = []
valid_names = []

for name in defra_names:

    pm25 = (
        march_week_2026['defra'][
            march_week_2026['defra']['Sensor_Name'] == name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    nox = (
        march_week_2026_NO['defra'][
            march_week_2026_NO['defra']['Sensor_Name'] == name
        ][['Timestamp', 'NOx', 'Time', 'Date']]
        .sort_values('Timestamp')
    )

    nox['NOx'] = pd.to_numeric(
        nox['NOx'],
        errors='coerce'
    )

    no2 = (
        march_week_2026_NO['defra'][
            march_week_2026_NO['defra']['Sensor_Name'] == name
        ][['Timestamp', 'NO2', 'Time', 'Date']]
        .sort_values('Timestamp')
    )

    no2['NO2'] = pd.to_numeric(
        no2['NO2'],
        errors='coerce'
    )

    # nearest timestamp match
    merged_nox = pd.merge_asof(
        pm25,
        nox,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_nox = merged_nox.dropna(subset=['PM2.5', 'NOx'])

    merged_no2 = pd.merge_asof(
        pm25,
        no2,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_no2 = merged_no2.dropna(subset=['PM2.5', 'NO2'])

    if not merged_nox.dropna().empty and not merged_no2.dropna().empty:
        valid_names.append((name, merged_nox, merged_no2))

In [ ]:
#| label: fig-pm25-no-defra
#| fig-cap: "Comparsion of PM2.5 and NOx and NO2 concentrations for DEFRA sensors over short time interval."

n_pairs = len(valid_names)

fig, axes = plt.subplots(
    nrows=n_pairs,
    ncols=2,
    figsize=(12, 4 * n_pairs),
    constrained_layout=True
)

# if only one row
if n_pairs == 1:
    axes = np.array([axes])

scatter_ref = None

for i, (name, merged_nox, merged_no2) in enumerate(valid_names):

    ax1, ax2 = axes[i]

        # Pearson correlation
    pearson_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='pearson').iloc[0, 1]
    pearson_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='spearman').iloc[0, 1]
    spearman_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'DEFRA': name,
    'Pearsons coeff NOx': pearson_corr_nox,
    'Spearmans coeff NOx': spearman_corr_nox,
    'Pearsons coeff NO2': pearson_corr_no2,
    'Spearmans coeff NO2': spearman_corr_no2
    })

    nox_plt = ax1.scatter(
    merged_nox['PM2.5'],
    merged_nox['NOx'],
    alpha=0.5,
    c='red',
    s=20
    )

    scatter_ref = nox_plt
    # 1:1 line
    lims = [
    merged_nox[['PM2.5', 'NOx']].min().min(),
    merged_nox[['PM2.5', 'NOx']].max().max()
]

    ax1.plot(lims, lims)

    ax1.set_xlabel('PM2.5')
    ax1.set_ylabel('NOx')
    ax1.set_title(f'{name}')
    ax1.grid(True)

    no2_plt = ax2.scatter(
        merged_no2['PM2.5'],
        merged_no2['NO2'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged_no2[['PM2.5', 'NO2']].min().min(),
    merged_no2[['PM2.5', 'NO2']].max().max()
]

    ax2.plot(lims, lims)

    ax2.set_xlabel('PM2.5')
    ax2.set_ylabel('NO2')
    ax2.set_title(f'{name}')
    ax2.grid(True)

#plt.tight_layout(rect=[0, 0, 0.92, 1])
plt.show()

In [ ]:
#| label: tbl-pm25-no-defra
#| tbl-cap: "Pearson's and Spearman's correlations for DEFRA sensor PM2.5 concentrations and NOx, NO2 concentrations over short time interval. Very low correlational coefficents for all sensors."

summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'DEFRA',
    'Pearsons coeff NOx',
    'Spearmans coeff NOx',
    'Pearsons coeff NO2',
    'Spearmans coeff NO2'
]]

summary_df = summary_df.sort_values(
    by='Pearsons coeff NOx',
    ascending=False
)

summary_df[['Pearsons coeff NOx', 'Spearmans coeff NOx', 'Pearsons coeff NO2', 'Spearmans coeff NO2']] = (
    summary_df[['Pearsons coeff NOx', 'Spearmans coeff NOx', 'Pearsons coeff NO2', 'Spearmans coeff NO2']].round(3)
)

Markdown(summary_df.to_markdown(index=False))

### Local sensors {#sec-local-no}

In @fig-pm25-no-local, we can see scatter plots for NO2 comparison with PM2.5 concentration for the Local sensors, they do not collect NOx data. Moderate correlational coefficets can be seen in @tbl-pm25-no-local which suggest there may be some traffic dependence or possibly other factors like temperature affecting PM2.5 concentrations.

In [ ]:
summary_results = []
valid_names = []

for name in local_names:

    pm25 = (
        march_week_2026['local'][
            march_week_2026['local']['Sensor_Name'] == name
        ][['Timestamp', 'PM2.5']]
        .sort_values('Timestamp')
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    no2 = (
        march_week_2026_NO['local'][
            march_week_2026_NO['local']['Sensor_Name'] == name
        ][['Timestamp', 'NO2', 'Time', 'Date']]
        .sort_values('Timestamp')
    )

    no2['NO2'] = pd.to_numeric(
        no2['NO2'],
        errors='coerce'
    )

    merged_no2 = pd.merge_asof(
        pm25,
        no2,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_no2 = merged_no2.dropna(subset=['PM2.5', 'NO2'])

    if not merged_no2.dropna().empty:
        valid_names.append((name, merged_no2))

In [ ]:
#| label: fig-pm25-no-local
#| fig-cap: "Comparisons of Local sensor PM2.5 and NO2 concentrations over short time interval."

n_plots = len(valid_names)
ncols = 2
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(12, 8),
    constrained_layout=True
)

axes = axes.flatten()
scatter_ref = no2_plt

for i, (name, merged_no2) in enumerate(valid_names):

    ax = axes[i]

    # Pearson correlation
    pearson_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_no2 = merged_no2[['PM2.5', 'NO2']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'Local': name,
    'Pearsons coeff NO2': pearson_corr_no2,
    'Spearmans coeff NO2': spearman_corr_no2
    })

    scatter_ref = ax.scatter(
        merged_no2['PM2.5'],
        merged_no2['NO2'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged_no2[['PM2.5', 'NO2']].min().min(),
    merged_no2[['PM2.5', 'NO2']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('NO2')
    ax.set_title(f'{name}')
    ax.grid(True)

# hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.show()

In [ ]:
#| label: tbl-pm25-no-local
#| tbl-cap: "Pearson's and Spearman's correlations for Local sensor PM2.5 concentrations and NO2 concentrations over short time interval. Moderate correlational coefficents for some sensors."

summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'Local',
    'Pearsons coeff NO2',
    'Spearmans coeff NO2'
]]

summary_df = summary_df.sort_values(
    by='Pearsons coeff NO2',
    ascending=False
)

summary_df[['Pearsons coeff NO2', 'Spearmans coeff NO2']] = (
    summary_df[['Pearsons coeff NO2', 'Spearmans coeff NO2']].round(3)
)

summary_df.set_index('Local', inplace=True)

summary_df

### UO-Mon sensors {#sec-mon-no}

In @fig-pm25-no-mon, scatter plots for NOx comparison with PM2.5 concentration for the UO-Mon sensors for now. In @tbl-pm25-no-mon, we see that some of the UO-Mon sensors show moderate positive correlation between PM2.5 and NOx, however the other half show little to no correlation.

In [ ]:
summary_results = []
valid_names = []

for name in monitor_names:

    pm25 = (
        march_week_2026['uo'][
            march_week_2026['uo']['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'PM2.5'})
    )

    pm25['PM2.5'] = pd.to_numeric(
        pm25['PM2.5'],
        errors='coerce'
    )

    nox = (
        march_week_2026_NO['uo'][
            march_week_2026_NO['uo']['Sensor_Name'] == name
        ][['Timestamp', 'Value']]
        .sort_values('Timestamp')
        .rename(columns={'Value': 'NOx'})
    )

    nox['NOx'] = pd.to_numeric(
        nox['NOx'],
        errors='coerce'
    )

    merged_nox = pd.merge_asof(
        pm25,
        nox,
        on='Timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('1min')
    )

    merged_nox = merged_nox.dropna(subset=['PM2.5', 'NOx'])

    if not merged_nox.dropna().empty:
        valid_names.append((name, merged_nox))

In [ ]:
#| label: fig-pm25-no-mon
#| fig-cap: "Comparisons of UO-Mon sensor PM2.5 and NOx concentrations over short time interval."

n_plots = len(valid_names)
ncols = 2
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows,
    ncols=ncols,
    figsize=(12, 8),
    constrained_layout=True
)

axes = axes.flatten()
scatter_ref = None

for i, (name, merged_nox) in enumerate(valid_names):
    
    ax = axes[i]

    # Pearson correlation
    pearson_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='pearson').iloc[0, 1]

    # Spearman correlation
    spearman_corr_nox = merged_nox[['PM2.5', 'NOx']].corr(method='spearman').iloc[0, 1]

    summary_results.append({
    'UO-Mon': name,
    'Pearsons coeff NOx': pearson_corr_nox,
    'Spearmans coeff NOx': spearman_corr_nox
    })

    scatter_ref = ax.scatter(
        merged_nox['PM2.5'],
        merged_nox['NOx'],
        alpha=0.5,
        c='red',
        s = 20
    )

    # 1:1 line
    lims = [
    merged_nox[['PM2.5', 'NOx']].min().min(),
    merged_nox[['PM2.5', 'NOx']].max().max()
]

    ax.plot(lims, lims)

    ax.set_xlabel('PM2.5')
    ax.set_ylabel('NOx')
    ax.set_title(f'{name}')
    ax.grid(True)

# hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.show()

In [ ]:
#| label: tbl-pm25-no-mon
#| tbl-cap: "Pearson's and Spearman's correlations for UO-Mon sensor PM2.5 concentrations and NOx concentrations over short time interval. Moderate correlational coefficents for some sensors."

summary_df = pd.DataFrame(summary_results)

summary_df = summary_df[[
    'UO-Mon',
    'Pearsons coeff NOx',
    'Spearmans coeff NOx'
]]

summary_df = summary_df.sort_values(
    by='Pearsons coeff NOx',
    ascending=False
)

summary_df[['Pearsons coeff NOx', 'Spearmans coeff NOx']] = (
    summary_df[['Pearsons coeff NOx', 'Spearmans coeff NOx']].round(3)
)

summary_df.set_index('UO-Mon', inplace=True)

summary_df